# Implementing a Client

**Guide for this lesson.** Do the exercise in the **starter** `cli-project/mcp_client.py`; the finished version is in `cli-project-complete/mcp_client.py`.

The server exposes tools, but the app can't use them until the **client** can (a) list them and (b) call them. The client has two parts:

- **`ClientSession`** — the actual connection to the server (from the MCP SDK).
- **`MCPClient`** — our thin wrapper that manages the session + cleanup (via `AsyncExitStack`) so callers don't have to.

This lesson fills in the two methods the app needs: **`list_tools`** and **`call_tool`**.

## What to modify

**File:** `cli-project/mcp_client.py` — replace the two `TODO` stubs (`list_tools` returning `[]`, `call_tool` returning `None`) with:

```python
async def list_tools(self) -> list[types.Tool]:
    result = await self.session().list_tools()
    return result.tools

async def call_tool(
    self, tool_name: str, tool_input: dict
) -> types.CallToolResult | None:
    return await self.session().call_tool(tool_name, tool_input)
```

Leave `list_prompts` / `get_prompt` / `read_resource` as their `TODO` stubs for now — they're wired in the **Defining/Accessing resources** and **Prompts in the client** lessons.

## How it works

- **`self.session()`** returns the live `ClientSession` (raising if not connected yet).
- **`list_tools`** calls the SDK's `session.list_tools()` and returns `result.tools` — a `ListToolsRequest` → `ListToolsResult` round-trip to the server (the message types from the *MCP clients* lesson).
- **`call_tool`** forwards the tool name + input (chosen by Claude) to `session.call_tool(...)` — a `CallToolRequest` → `CallToolResult`.

That's the whole client surface the app needs. `MCPClient` hides the connection/cleanup; `list_tools`/`call_tool` are one-liners over the session.

## How it plugs into the app

In `core/tools.py`, the `ToolManager` calls **`list_tools`** across all connected clients to build the `tools` list sent to Claude, and calls **`call_tool`** when Claude returns a `tool_use` block. So the moment these two methods work, the existing tool-use loop (`core/chat.py`) can read and edit documents end to end:

```
user question
  -> client.list_tools()  -> tools sent to Claude
  -> Claude asks for read_doc_contents
  -> client.call_tool("read_doc_contents", {"doc_id": "report.pdf"})
  -> result back to Claude
  -> Claude answers the user
```

## Testing — run the app and chat

The client's real test is end-to-end (it spawns the server as a subprocess), so test it in a **terminal**, not this notebook.

```powershell
.\.venv\Scripts\Activate.ps1
cd 01-building-with-the-claude-api\07-mcp\cli-project-complete   # already has the methods
python main.py
```

Then ask:

> **What is the contents of the report.pdf document?**

Claude should call `read_doc_contents` and answer with the 20m condenser tower text from the server's `docs`. Try an edit too — *"In report.pdf, change '20m' to '35m'"* — then ask for the contents again to confirm.

- Run in **`cli-project-complete/`** to test immediately.
- Run in **`cli-project/`** only after you've added the two methods there (before that, the bot answers general questions but says it has no way to access documents).

> There's also a `python mcp_client.py` test harness at the bottom of the file — the complete version just connects (`pass`); to see tools, you'd add a `print(await client.list_tools())`. The full-app chat above is the more useful check.

## Where this leaves us

The chatbot is now genuinely useful: it can **read and edit the documents** on the MCP server through the client. Everything after this extends the *kinds* of things the server offers and the client consumes:

- **Defining resources** / **Accessing resources** — expose doc ids + contents as MCP *resources*, and implement `read_resource` on the client.
- **Defining prompts** / **Prompts in the client** — server-provided prompt templates, consumed via `list_prompts` / `get_prompt`.

Next: **Defining resources**.